# Distributed Training Fundamentals

> How does a training script go from one GPU to many? The easiest answer is Accelerate: the model, the data, and the training loop barely change, and a single launch command turns the same script into DDP, FSDP, or DeepSpeed training. Get it running first, then come back for the mechanics.
>
> The first problem you hit after scaling out is memory: a 7B model under AdamW and mixed precision needs roughly 112 GB of resident state, more than one 80 GB A100. ZeRO's three stages answer "which states get sharded across GPUs"; DeepSpeed and FSDP are its two mainstream implementations.
>
> When models reach tens of billions of parameters and clusters reach thousands of GPUs, data-parallel variants are no longer enough. Megatron-LM's 3D parallelism — the combination of tensor, pipeline, and data parallelism — is the industrial standard for pretraining from scratch.
>
> This chapter's route: first write a script with Accelerate that runs on one GPU and launches on many, then work out the memory budget and how to pick the fastest ZeRO config, and finally survey Megatron and multi-node training.

Start from the fastest path to a working setup: with the right script structure, you can finish multi-GPU training setup on a single machine today. ZeRO configs for tight memory, and 3D parallelism for larger scale, are questions to answer only after things are running.

In [ ]:
# All imports for this chapter live in the first code cell
import json
import os
import shutil

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from accelerate import Accelerator

torch.manual_seed(42)
print("torch:", torch.__version__)

## 1. Writing the Training Script with Accelerate

This section rewrites the mini Trainer from "First Pretraining and Fine-tuning" into a multi-GPU version. With raw PyTorch, switching backends means editing code: DDP needs `torchrun` plus a DistributedSampler around the DataLoader, FSDP needs a wrapper around the model, DeepSpeed needs an initialized engine and a JSON config.

Accelerate unifies these differences behind a single `Accelerator` object. It is not itself a distributed algorithm: in DDP mode the backend is torch.distributed, in FSDP mode it is PyTorch FSDP, and in DeepSpeed mode it is DeepSpeed.

The rewrite has three steps: model and data, Accelerator initialization, and the training loop. Launch commands and effective batch size come in the next section, checkpoints in Section 5.

### 1.1 Model and Data

The model is a tiny causal language model: an embedding layer followed by an lm_head. The data is a set of token sequences with fixed patterns. Although everything is small, the interfaces match a real setup: the Dataset yields single samples, the DataLoader forms batches, and the model takes `input_ids` and `labels` and returns a loss. This code is identical to the single-GPU script — no distributed changes at all.

In [ ]:
# === Model and data: identical to the single-GPU script ===
class TinyCausalLM(nn.Module):
    """Tiny causal LM: embedding feeding an lm_head."""

    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids, labels=None):
        """
        Token ids in, logits out; returns the loss too when labels are given.

        input_ids: [batch, seq_len], sample with the last token dropped
        labels:    [batch, seq_len], shifted-left next tokens
        """
        logits = self.lm_head(self.embedding(input_ids))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        return {"loss": loss, "logits": logits}


class ToyTextDataset(Dataset):
    """Each sample is a short token sequence: [1] starts, [2] ends, fixed pattern inside."""

    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        ids = self.sequences[index]
        # input is the first n-1 tokens, supervision is the last n-1 (next-token prediction)
        return {
            "input_ids": torch.tensor(ids[:-1]),
            "labels": torch.tensor(ids[1:]),
        }


def simple_collate(features):
    """Stack a list of sample dicts into whole-batch tensors."""
    return {
        "input_ids": torch.stack([f["input_ids"] for f in features]),
        "labels": torch.stack([f["labels"] for f in features]),
    }


# Two alternating fixed patterns; the model can learn rules like "after 1 comes 3"
sequences = [
    [1, 3, 4, 5, 6, 2] if i % 2 == 0 else [1, 3, 4, 6, 5, 2]
    for i in range(16)
]

dataset = ToyTextDataset(sequences)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=simple_collate)

model = TinyCausalLM(vocab_size=9, hidden_size=32)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.05)

batch = next(iter(dataloader))
print("One batch shapes:", {k: tuple(v.shape) for k, v in batch.items()})

The output shows both `input_ids` and `labels` have shape `[4, 5]`: 4 samples, 5 tokens each. So far there is nothing distributed in this code.

### 1.2 Initializing the Accelerator

The multi-GPU adaptation starts with creating an `Accelerator`. Its `prepare` method takes the training trio (model, optimizer, dataloader) and wraps them for the current backend. On GPUs `mixed_precision` is usually set to `"bf16"`; this CPU demo disables it:

In [ ]:
# === Create the Accelerator and prepare ===
accelerator = Accelerator(
    gradient_accumulation_steps=2,  # 2 micro-batches add up to one full step
    mixed_precision="no",           # no mixed precision on CPU; use "bf16" on real GPUs
)

model, optimizer, dataloader = accelerator.prepare(model, optimizer, dataloader)

print("distributed_type:", accelerator.distributed_type)
print("num_processes:   ", accelerator.num_processes)
print("process_index:   ", accelerator.process_index)
print("is_main_process: ", accelerator.is_main_process)

In single-process execution, `num_processes` is 1, `process_index` is 0, and `is_main_process` is True. After launching with `accelerate launch --num_processes 8`, each process gets a different `process_index` and a different data shard — and this code needs no modification at all.

### 1.3 The Training Loop

Compared with the single-GPU loop, exactly three things change:

- `with accelerator.accumulate(model)` manages gradient accumulation: until 2 micro-batches have been collected, `optimizer.step()` inside the block does not actually update the weights;
- `accelerator.backward(loss)` replaces `loss.backward()` — under DDP it synchronizes gradients across GPUs, under DeepSpeed ZeRO it handles gradient sharding and reduction;
- log output is gated by `is_main_process`, so N processes don't each print the same line.

In [ ]:
# === Full training loop: drop it into train.py unchanged ===
num_epochs = 8

for epoch in range(num_epochs):
    total_loss, num_steps = 0.0, 0

    for batch in dataloader:
        with accelerator.accumulate(model):
            outputs = model(batch["input_ids"], labels=batch["labels"])
            accelerator.backward(outputs["loss"])
            optimizer.step()
            optimizer.zero_grad()

        if accelerator.is_main_process:
            total_loss += outputs["loss"].item()
            num_steps += 1

    if accelerator.is_main_process and (epoch + 1) % 2 == 0:
        print(f"epoch {epoch + 1:02d} | train_loss = {total_loss / num_steps:.4f}")

The loss drops from about 0.65 to about 0.42 (exact values vary slightly with random init), the same behavior as single-GPU training. The very same loop is single-GPU training in single-process mode and 8-GPU data parallelism when launched with 8 processes.

## 2. Launching Multi-GPU and the Effective Batch Size

`gradient_accumulation_steps` combines with two other numbers to decide how many samples each parameter update sees:

$$\text{effective batch} = \text{micro-batch} \times \text{number of GPUs} \times \text{accumulation steps}$$

The example in 1.3 is 4 × 1 × 2 = 8. In a real project, micro-batch 4 on 8 GPUs with 4 accumulation steps gives an effective batch of 128: each GPU only holds activations for 4 samples at a time, while the update is based on gradients from 128 samples. Memory consumption is set by the micro-batch; training behavior is set by the effective batch.

Two other common launch parameters deserve a note. `num_processes` is the total GPU count; every process runs the same script on a different data shard. `mixed_precision` defaults to `bf16` in industry today: A100/H100 support it natively, its numeric range matches FP32, and it needs none of FP16's loss-scaling machinery (see the appendix "Mixed Precision Training and Loss Scaling").

### 2.1 Launch Commands

Once the script is done, switching backends only means switching the launch command. In the commands below, `train.py` is the code from the previous few sections combined:

In [ ]:
# === Launch commands: single GPU, multi GPU, DeepSpeed — one script ===
print("Single process (equivalent to python train.py):")
print("  $ accelerate launch train.py")
print()
print("8 GPUs on one machine:")
print("  $ accelerate launch --num_processes 8 --multi_gpu train.py")
print()
print("DeepSpeed ZeRO-3 backend (with the ds_config.json from 4.1):")
print("  $ accelerate launch --use_deepspeed --ds_config_file ds_config.json \\")
print("      --num_processes 8 train.py")
print()
print("You can also run accelerate config (interactive Q&A) to generate a config file,")
print("then always launch with accelerate launch --config_file xxx.yaml train.py.")

Switching from DDP to FSDP or DeepSpeed changes only the launch command (or the config file produced by `accelerate config`); the code from 1.1–1.3 stays untouched.

## 3. Why You OOM: The Single-GPU Memory Budget

Under AdamW with mixed precision, each parameter costs 16 bytes of GPU memory across five components:

| State | Precision | Size |
|:---|:---|:---|
| Parameters | FP16 | 2 bytes |
| Gradients | FP16 | 2 bytes |
| Master weights | FP32 | 4 bytes |
| AdamW first moment | FP32 | 4 bytes |
| AdamW second moment | FP32 | 4 bytes |

As a formula, with $P$ the parameter count:

$$16P = 2P + 2P + 4P + 4P + 4P$$

This cost is independent of sequence length; it depends only on the parameter count.

In [ ]:
# === Single-GPU memory budget: 7B model ===
P = 7e9  # 7B parameters

param_fp16  = 2 * P  # FP16 parameters
grad_fp16   = 2 * P  # FP16 gradients
master_fp32 = 4 * P  # FP32 master weights
adam_m      = 4 * P  # AdamW first moment
adam_v      = 4 * P  # AdamW second moment

fixed_bytes = param_fp16 + grad_fp16 + master_fp32 + adam_m + adam_v
fixed_gb = fixed_bytes / 1e9

print(f"Parameters (FP16):     {param_fp16 / 1e9:.1f} GB")
print(f"Gradients (FP16):      {grad_fp16 / 1e9:.1f} GB")
print(f"Master weights (FP32): {master_fp32 / 1e9:.1f} GB")
print(f"AdamW m (FP32):        {adam_m / 1e9:.1f} GB")
print(f"AdamW v (FP32):        {adam_v / 1e9:.1f} GB")
print(f"Fixed total:           {fixed_gb:.0f} GB (= 16 x P bytes)")

A 7B model has a fixed cost of 112 GB, of which the three optimizer-related parts (master weights and the two moments) total 84 GB — the largest share.

The most direct use of multiple GPUs is Distributed Data Parallel (DDP): every GPU holds a full replica of the model, the dataset is split into N shards for N GPUs, each does its own forward and backward pass, and an all-reduce averages the gradients. DDP throughput scales nearly linearly with GPU count, but memory is not saved at all: of the 896 GB across 8 GPUs, 7/8 are identical redundant copies.

## 4. The Three ZeRO Stages

ZeRO (Zero Redundancy Optimizer) progressively shards the redundant 16P bytes of state on each GPU across N GPUs; after sharding, the group of GPUs is still logically one full model. The three stages are cumulative — each one shards one more piece:

- **Stage 1** shards optimizer state, 12P becomes 12P/N; each GPU updates only its own 1/N of the parameters;
- **Stage 2** additionally shards gradients, 2P becomes 2P/N; during backprop a reduce-scatter leaves each GPU with only its own slice of gradients;
- **Stage 3** additionally shards parameters, 2P becomes 2P/N; forward and backprop all-gather each layer on demand and release it after use.

In [ ]:
# === ZeRO three-stage memory: 7B model x 8 GPUs ===
P = 7e9
N = 8

print(f"{'Setup':<16}{'params':>8}{'grads':>8}{'optimizer':>10}{'per-GPU':>10}{'vs DDP':>10}")
print("-" * 64)
configs = [
    ("DDP",          2 * P,       2 * P,       12 * P),
    ("ZeRO Stage 1", 2 * P,       2 * P,       12 * P / N),
    ("ZeRO Stage 2", 2 * P,       2 * P / N,   12 * P / N),
    ("ZeRO Stage 3", 2 * P / N,   2 * P / N,   12 * P / N),
]
ddp_bytes = 16 * P
for name, p, g, o in configs:
    total = p + g + o
    print(f"{name:<16}{p/1e9:>7.1f} {g/1e9:>7.1f} {o/1e9:>9.1f} "
          f"{total/1e9:>8.1f} GB {total/ddp_bytes*100:>8.1f}%")

From DDP to Stage 3, per-GPU memory drops from 112 GB to 14 GB — an 8× reduction. The price is more communication: the higher the stage, the more shards must be passed around during forward and backward.

In practice, Stage 2's communication overhead is close to DDP's, making it the best value; the inter-node-bandwidth-sensitive Stage 3 is used only when the parameters genuinely don't fit.

### 4.1 OOM Escalation Ladder and the DeepSpeed Config

When you hit OOM, escalate in this order, from cheapest to most expensive:

1. Shrink the micro-batch and increase gradient accumulation;
2. Upgrade ZeRO Stage 2 to Stage 3;
3. Enable `offload_optimizer`, moving optimizer state to CPU RAM (biggest saving, but CPU↔GPU transfers cost time);
4. Enable `offload_param`, moving parameters to CPU too (Stage 3 only);
5. Offload to NVMe storage (ZeRO-Infinity).

These knobs live in DeepSpeed's JSON config file, which the launch command from the previous section passes via `--ds_config_file`:

In [ ]:
# === DeepSpeed ZeRO config: the fields you actually touch ===
deepspeed_config = {
    "train_micro_batch_size_per_gpu": 4,
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,                    # 1 shards optimizer state / 2 adds grads / 3 adds params
        "offload_optimizer": {         # move optimizer state to CPU RAM
            "device": "cpu",
            "pin_memory": True,        # pinned memory: faster CPU->GPU copies
        },
        "offload_param": {             # move parameters to CPU too (stage 3 only)
            "device": "none",
        },
        "overlap_comm": True,          # overlap communication with compute to hide latency
        "contiguous_gradients": True,  # store gradients in contiguous blocks, fewer fragments
        "reduce_bucket_size": 5e8,     # gradient bucket size (bytes); big buckets are efficient but cost memory
    },
}

print("ds_config.json:")
print(json.dumps(deepspeed_config, indent=2))

In practice you only touch `stage` and the two offloads; the rest can keep their defaults:

| Parameter | Effect | When to touch |
|:---|:---|:---|
| `stage` (1/2/3) | Shard optimizer state / grads / params | Upgrade when memory runs out; 2 and 3 are common |
| `offload_optimizer` | Offload optimizer state to CPU | When Stage 3 still isn't enough; speed for memory |
| `offload_param` | Offload parameters to CPU too | Model too big for GPU; Stage 3 only |
| `overlap_comm` | Overlap communication with compute | On by default; verify it's on when inter-GPU links are slow |
| `contiguous_gradients` | Contiguous gradient storage | On by default; usually leave it |
| `reduce_bucket_size` | Gradient bucket size (bytes) | Raise when communication is the bottleneck, at a memory cost |

If the backend is FSDP instead, the concepts map one-to-one: `FULL_SHARD` equals ZeRO-3, `SHARD_GRAD_OP` equals ZeRO-2. Choosing between them is mostly an engineering preference: DeepSpeed is config-driven with a more complete offload ecosystem; FSDP is maintained by PyTorch proper and supports new hardware faster. The Accelerate launch commands from Section 2 switch between them without code changes.

### 4.2 Going Fastest: Don't Turn On ZeRO By Default

ZeRO and offloading both trade speed for memory. The principle for fastest training is simple: **don't shard what fits, don't offload what fits** — max out the micro-batch and keep the GPUs busy. That principle gives this decision table:

| Memory situation | Recommended setup | Speed cost |
|:---|:---|:---|
| Fits comfortably | Plain DDP + bf16 + FlashAttention-2 | Baseline, fastest |
| Optimizer state doesn't fit | ZeRO Stage 2 | Communication close to DDP, small loss |
| Parameters don't fit | ZeRO Stage 3 | More communication; notably slower across nodes |
| Single GPU never fits | Stage 3 + offload optimizer | CPU↔GPU transfers, clearly slower |
| Still not enough | Add offload param | Slowest, last resort |

A practical workflow: first get it running with plain DDP and watch memory (`nvidia-smi` or Accelerate's tracker); if peak usage is below 70%, increase the micro-batch before enabling anything new; if you OOM, step down the table one row at a time. Measure first, then tune — don't sacrifice speed just to look sophisticated.

## 5. Saving and Restoring Checkpoints

Real training runs last days to weeks and must survive interruption. Accelerate provides a matched pair: `save_state` saves the model, optimizer, and DataLoader progress together; `load_state` restores them in a new process. Call both on every process; Accelerate handles multi-process consistency.

In [ ]:
# === Checkpoint demo: save -> simulate a fresh process restoring -> verify weights ===
ckpt_dir = "_ckpt_demo"

accelerator.save_state(ckpt_dir)
print("Files saved by save_state:")
for name in sorted(os.listdir(ckpt_dir)):
    print(" ", name)

# Simulate "restore on another machine": new process = new Accelerator + untrained new model
accelerator_restored = Accelerator(
    gradient_accumulation_steps=2,
    mixed_precision="no",
)
model_restored = TinyCausalLM(vocab_size=9, hidden_size=32)
optimizer_restored = torch.optim.AdamW(model_restored.parameters(), lr=0.05)
model_restored, optimizer_restored = accelerator_restored.prepare(
    model_restored, optimizer_restored
)
accelerator_restored.load_state(ckpt_dir)

w_trained = accelerator.unwrap_model(model).embedding.weight
w_restored = accelerator_restored.unwrap_model(model_restored).embedding.weight
assert torch.allclose(w_trained, w_restored), "restored weights should match the saved ones"

shutil.rmtree(ckpt_dir)  # clean up the demo directory

`save_state` wrote three kinds of files: model weights, optimizer state, and RNG state. After `load_state` the weights match the saved ones bit for bit, AdamW's first and second moments are restored too, and training continues from the checkpoint seamlessly.

## 6. Common Fine-tuning Settings

Finally, the high-frequency settings used together with distributed training — nearly every fine-tuning script uses some of them:

- **Gradient accumulation**: trades time for space when a big batch doesn't fit in memory;
- **Gradient checkpointing**: don't store intermediate activations during backprop, recompute them instead; activation memory drops 60%+, overall speed drops about 30%;
- **BF16 mixed precision**: parameters and activations in BF16; roughly 2x faster compute and half the memory;
- **FlashAttention-2**: attention without materializing the full attention matrix; long sequences benefit in both memory and speed (see the appendix "FlashAttention's Blocked Computation");
- **8-bit optimizer**: squeezes AdamW state from 12P to 3P bytes with negligible precision loss.

The corresponding knobs in the HuggingFace ecosystem:

| Setting | Problem it solves | Typical usage |
|:---|:---|:---|
| Gradient accumulation | Batch too big for memory | `gradient_accumulation_steps=8` |
| Gradient checkpointing | Activations eat memory | `gradient_checkpointing=True` |
| BF16 mixed precision | Compute speed and memory | `bf16=True` |
| FlashAttention-2 | Long-sequence attention slow and memory-hungry | `attn_implementation='flash_attention_2'` |
| 8-bit optimizer | 12P optimizer state too large | `optim='adamw_bnb_8bit'` |

Typical combinations: single-GPU LoRA fine-tuning of a 7B model uses bf16 + gradient accumulation + FlashAttention-2; multi-GPU full fine-tuning of a 70B model uses bf16 + ZeRO-3 + offload + gradient checkpointing.

## 7. Advanced Survey: Megatron, 3D Parallelism, and Multi-Node

ZeRO's two mainstream implementations, DeepSpeed and PyTorch FSDP, are essentially data-parallel variants. When models reach tens of billions of parameters and clusters reach thousands of GPUs, these hit a wall: ZeRO-3 all-gathers parameters at every layer, and cross-node traffic balloons with scale.

The industrial standard for pretraining from scratch is Megatron-LM's 3D parallelism — a combination of three kinds of parallelism:

- **Tensor Parallelism (TP)** splits each layer's matrix multiplications across GPUs. Communication is one all-reduce per layer and latency-sensitive, so TP stays inside a node, exploiting NVLink bandwidth;
- **Pipeline Parallelism (PP)** splits the model into segments by layer; different segments run on different nodes in relay. Communication is small (only activations at segment boundaries), a good fit for high-latency cross-node links;
- **Data Parallelism (DP)** replicates model copies in the outermost dimension, with ZeRO sharding the redundant state.

The rule of thumb when laying out a cluster: keep TP within a node, let PP cross nodes, and give the remaining GPUs to DP.

Megatron differs from DeepSpeed / FSDP in positioning: those optimize memory for data-parallel training, while Megatron is a full pretraining framework with model definition, data loading, loss computation, and checkpoints built in. Its common arguments:

| Argument | Meaning |
|:---|:---|
| `--tensor-model-parallel-size 8` | TP degree = 8; each layer's matmuls split 8 ways across the node's 8 GPUs |
| `--pipeline-model-parallel-size 16` | PP degree = 16; the model split into 16 segments relaying across nodes |
| `--global-batch-size 1024` | Samples per full step; the framework derives gradient accumulation steps |
| `--sequence-parallel` | Also shard the sequence dimension; pairs with TP to save activation memory |
| `--recompute-activations` | Recompute activations (gradient checkpointing); a pretraining standard |

Tool choice across three typical scenarios:

| Scenario | Tool |
|:---|:---|
| Pretraining 70B+ from scratch, thousands of GPUs | Megatron family (3D parallelism) |
| Pretraining 7B–13B from scratch | Accelerate + FSDP is enough |
| Fine-tuning, up to a few hundred GPUs | Accelerate + ZeRO / FSDP |

(For the internals and hand calculations of TP / PP, see the appendix "Five Ways to Shard a Large Model"; PyTorch's official torchtitan takes a lightweight FSDP + TP route suited to medium scale.)

### 7.1 Launching Multi-Node

Every command so far runs on one machine. Scaling to multiple machines requires two concepts:

- **Node**: one machine, usually with 8 GPUs;
- **Process**: one process per GPU; two 8-GPU machines have 16 processes in total.

Multi-node launch needs a "main address" so all nodes can find each other: the main node (`machine_rank 0`) fixes an IP and port, and every other node launches with that address. The training code still doesn't change:

In [ ]:
# === Multi-node: two machines, 8 GPUs each, 16 processes total ===
# On the main node (assume IP 10.0.0.1):
print("accelerate launch --multi_gpu --num_processes 16 "
      "--main_process_ip 10.0.0.1 --main_process_port 29500 "
      "--machine_rank 0 train.py")
print()
# On the other machine: only --machine_rank changes to 1
print("accelerate launch --multi_gpu --num_processes 16 "
      "--main_process_ip 10.0.0.1 --main_process_port 29500 "
      "--machine_rank 1 train.py")
print()
# Using PyTorch's native torchrun is equivalent:
print("torchrun --nnodes 2 --nproc-per-node 8 --rdzv-endpoint=10.0.0.1:29500 train.py")
print()
print("Key observation: both machines run the same command, only machine_rank differs;")
print("the training script itself never mentions the number of machines.")

## Summary

- The multi-GPU deltas versus single-GPU concentrate in four spots: Accelerator, prepare, accumulate/backward, is_main_process
- Effective batch = micro-batch × GPU count × gradient accumulation steps; memory is set by the micro-batch, training behavior by the effective batch
- A 7B full-training run costs about 112 GB fixed, i.e. 16 bytes × parameter count; DDP doesn't reduce it
- ZeRO's three stages shard optimizer state, gradients, and parameters respectively — more sharding saves more memory but costs more communication
- Fastest-config priority: skip ZeRO while memory fits; Stage 2 when tight; Stage 3 + offload only when parameters don't fit
- OOM escalation order: gradient accumulation -> Stage 2 -> Stage 3 -> offload optimizer -> offload param
- Checkpoints use save_state / load_state, restoring model and optimizer state together
- 3D parallelism rule of thumb: TP within a node, PP across nodes, remaining GPUs to DP
- Tool choice: Megatron family for large-scale pretraining from scratch; Accelerate + ZeRO/FSDP for fine-tuning

## Exercises

> You can ask an AI to explain the ideas or check your direction, but don't have it "solve the exercise" for you.

**Exercise 1: ZeRO Stage 2 per-GPU memory on 4 GPUs**

A 7B model trained with AdamW on 4 GPUs. What is the fixed memory per GPU under ZeRO Stage 2, in GB?

Hint: Stage 2 shards gradients and optimizer state (14P split 4 ways); parameters stay fully replicated per GPU (2P).

In [ ]:
# Exercise 1: ZeRO Stage 2 per-GPU memory on 4 GPUs
P = 7e9
N = 4

# TODO: compute Stage 2 per-GPU memory (in GB)
# parameters fully replicated + (gradients + optimizer state) sharded N ways
s2_per_card_gb = (2 * P + 14 * P / N) / 1e9

assert s2_per_card_gb is not None, "compute Stage 2 per-GPU memory first"
expected = (2 * P + 14 * P / N) / 1e9
assert abs(s2_per_card_gb - expected) < 0.1, f"should be {expected:.1f} GB"
print("Exercise 1 passed:")
print(f"   Stage 2 + 4 GPUs + 7B: {s2_per_card_gb:.1f} GB per GPU")
print(f"   That saves {112 - s2_per_card_gb:.1f} GB vs DDP's 112 GB, at nearly no communication cost.")

**Exercise 2: write a DeepSpeed config for a memory emergency**

Fill in `deepspeed_config` below with: ZeRO Stage 3, optimizer state offloaded to CPU, and communication/compute overlap enabled.

Hint: that's the three fields `zero_optimization.stage`, `offload_optimizer.device`, `overlap_comm`.

In [ ]:
# Exercise 2: DeepSpeed config for a memory emergency
deepspeed_config = {
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu"},
        "overlap_comm": True,
    },
}

# Verify
zero = deepspeed_config.get("zero_optimization", {})
assert zero.get("stage") == 3, "stage should be 3"
assert zero.get("offload_optimizer", {}).get("device") == "cpu", "offload_optimizer.device should be 'cpu'"
assert zero.get("overlap_comm") is True, "overlap_comm should be True"

print("Exercise 2 passed:")
print(json.dumps(deepspeed_config, indent=2))
print()
print("This config = the first-line setup for the tightest memory: shard params/grads/optimizer + optimizer on CPU.")

**Exercise 3: compute the effective batch size**

A pretraining job: micro-batch 2, 32 GPUs, gradient accumulation 8 steps. How many samples does one full optimization step use?

Hint: multiply the three numbers — the real-world version of the three factors in Section 2's formula.

In [ ]:
# Exercise 3: compute the effective batch size
micro_batch = 2
num_gpus = 32
grad_accum = 8

# TODO: compute the effective batch size
effective_batch = micro_batch * num_gpus * grad_accum

assert effective_batch is not None, "compute the effective batch size first"
expected = micro_batch * num_gpus * grad_accum
assert effective_batch == expected, f"should be {expected}"
print("Exercise 3 passed:")
print(f"   Effective batch = {micro_batch} x {num_gpus} x {grad_accum} = {effective_batch}")
print(f"   Each GPU only holds activations for {micro_batch} samples, yet achieves a batch of {effective_batch}.")

## References

- Rajbhandari et al., [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054), 2020
- [HuggingFace Accelerate documentation](https://huggingface.co/docs/accelerate/)
- [DeepSpeed ZeRO configuration docs](https://www.deepspeed.ai/docs/config-json/)
- Shoeybi et al., [Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism](https://arxiv.org/abs/1909.08053), 2019
- [NVIDIA Megatron-LM GitHub](https://github.com/NVIDIA/Megatron-LM)